> **AI attribution.** Roughly half-and-half. The 80/10/10 split scaffolding and the two baseline functions (majority class, random-weighted) are mine. The 7-config hyperparameter sweep and the LR-vs-NB head-to-head loop were drafted by Claude based on my specification — I chose the C and α grids, the metric set, and which model to save. I read the sweep results and picked the winning config myself. See [ATTRIBUTION.md](../ATTRIBUTION.md) for the full breakdown.

# Modeling — Baselines + Classifiers

Now that EDA + preprocessing are done.
1. Set up a shared 80/10/10 split so every model sees the same data
2. Run two baselines: majority class + random-weighted
3. Logistic regression + naive bayes with a small hyperparameter sweep

Reusing `src/preprocess.py` for cleaning and `src/features.py` for TF-IDF to keep the notebook thin.

In [1]:
import sys
sys.path.insert(0, "..")

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from src.preprocess import clean_text
from src.features import build_tfidf_vectorizer
from src.classify import split_data, train_model, evaluate_model

RANDOM_STATE = 42

## Load + Clean

In [2]:
df = pd.read_csv("../data/processed/songs_labeled.csv")
print(df.shape)
df["mood"].value_counts()

(76595, 10)


mood
Hype        41831
Sad         12513
Romantic     9000
Angry        6632
Calm         6619
Name: count, dtype: int64

In [3]:
# clean_text takes a second on 76k rows but it's fine
df["lyrics_clean"] = df["lyrics"].map(clean_text)
df[["name", "mood", "lyrics_clean"]].head(2)

,name,mood,lyrics_clean
0,Blood,Hype,bound restless yearning satisfied m aching wor...
1,The Ugly Duckling,Calm,gods gods know stronger native proverb east su...


## 80 / 10 / 10 split

Stratified so every mood shows up in train/val/test. Doing two `train_test_split` calls — first splits off test (10%), then splits train-val (90%) into train (88.9% of that = 80%) and val (11.1% = 10%).

`random_state=42` everywhere so the split is reproducible across notebooks.

In [4]:
X = df["lyrics_clean"]
y = df["mood"]

# using src.classify.split_data so this same split logic is reused
# in the evaluation notebook later
X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    X, y, val_size=0.1, test_size=0.1, random_state=RANDOM_STATE,
)

print(f"train: {len(X_train):>6} ({len(X_train)/len(df):.1%})")
print(f"val:   {len(X_val):>6} ({len(X_val)/len(df):.1%})")
print(f"test:  {len(X_test):>6} ({len(X_test)/len(df):.1%})")

train:  61275 (80.0%)
val:     7660 (10.0%)
test:    7660 (10.0%)


## Baselines

Two dumb predictors. Every real model has to beat both or it's not earning its keep:

- **Majority class** — just predict whatever mood shows up most. With Hype at ~55% of the data this will have OK accuracy but awful macro F1.
- **Random weighted** — sample class labels proportional to their training frequency. More honest baseline for imbalanced data since it spreads predictions across all classes.

In [5]:
def majority_class_baseline(y_train: pd.Series, y_test: pd.Series) -> dict:
    """Predict the most frequent training class for every test sample."""
    top_class = y_train.value_counts().idxmax()
    preds = np.full(len(y_test), top_class)
    return {
        "pred_class": top_class,
        "accuracy": accuracy_score(y_test, preds),
        "macro_f1": f1_score(y_test, preds, average="macro"),
    }


def random_weighted_baseline(y_train: pd.Series, y_test: pd.Series, seed: int = RANDOM_STATE) -> dict:
    """Predict class labels sampled in proportion to training-set frequencies."""
    rng = np.random.default_rng(seed)
    probs = y_train.value_counts(normalize=True)
    preds = rng.choice(probs.index.to_numpy(), size=len(y_test), p=probs.to_numpy())
    return {
        "accuracy": accuracy_score(y_test, preds),
        "macro_f1": f1_score(y_test, preds, average="macro"),
    }

In [6]:
maj = majority_class_baseline(y_train, y_test)
rnd = random_weighted_baseline(y_train, y_test)

print(f"majority class  ({maj['pred_class']!r:>10}) : acc={maj['accuracy']:.3f}  macro F1={maj['macro_f1']:.3f}")
print(f"random weighted                : acc={rnd['accuracy']:.3f}  macro F1={rnd['macro_f1']:.3f}")

majority class  (    'Hype') : acc=0.546  macro F1=0.141
random weighted                : acc=0.361  macro F1=0.201


In [7]:
baseline_table = pd.DataFrame([
    {"model": "majority class",  "accuracy": maj["accuracy"],  "macro_f1": maj["macro_f1"]},
    {"model": "random weighted", "accuracy": rnd["accuracy"], "macro_f1": rnd["macro_f1"]},
])
baseline_table.round(3)

,model,accuracy,macro_f1
0,majority class,0.546,0.141
1,random weighted,0.361,0.201


### Baseline results

| model | accuracy | macro F1 |
|---|---|---|
| majority class | 0.546 | 0.141 |
| random weighted | 0.361 | 0.201 |

Expected picture plays out: majority-class gets 0.546 accuracy (Hype is ~55% of the test set) but macro F1 tanks to 0.141 because it's only ever right on one class. Random-weighted is worse on raw accuracy (0.361) but has a less degenerate macro F1 (0.201) because it spreads predictions across all five moods.

Any real model trained later needs to beat **both** of these — the one that really matters is macro F1, since the classes are imbalanced and accuracy can be gamed by just predicting Hype.

## TF-IDF features

Fitting TF-IDF **only on train** — the vectorizer sees no val/test text, otherwise the vocabulary leaks information about the held-out songs. Unigrams + bigrams, 10k feature cap (same settings as `src/features.py` default).

In [8]:
tfidf = build_tfidf_vectorizer()
Xtr = tfidf.fit_transform(X_train)
Xva = tfidf.transform(X_val)
Xte = tfidf.transform(X_test)
print("train:", Xtr.shape, "| val:", Xva.shape, "| test:", Xte.shape)

train: (61275, 20000) | val: (7660, 20000) | test: (7660, 20000)


## Hyperparameter Sweep

Two architectures, a handful of configs each. Class imbalance is bad enough that I'm keeping `class_weight='balanced'` on LR throughout — task 2 showed that's worth about +0.014 macro F1 and NB doesn't support it.

- **Logistic Regression** — L2 penalty, `C ∈ {0.01, 0.1, 1.0, 10.0}`. C is the inverse of regularization strength, so low C = lots of regularization, high C = essentially unregularized.
- **Multinomial NB** — `α ∈ {0.01, 0.1, 1.0}`. α is the Laplace smoothing; smaller α trusts the training counts more, bigger α smooths them toward uniform.

7 total configs. All scored on the same val set.

In [9]:
results = []
fitted = {}  # keep fitted models so I can re-evaluate the winner on test later

# Logistic Regression sweep
for C in [0.01, 0.1, 1.0, 10.0]:
    name = f"LR (C={C})"
    m = train_model(
        Xtr, y_train, LogisticRegression,
        C=C, penalty="l2", class_weight="balanced",
        max_iter=2000, n_jobs=-1, random_state=RANDOM_STATE,
    )
    val_metrics = evaluate_model(m, Xva, y_val)
    results.append({
        "model": "LogisticRegression",
        "params": f"C={C}, penalty=l2, class_weight=balanced",
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
    })
    fitted[name] = m
    print(f"{name:>14}: acc={val_metrics['accuracy']:.3f}  F1={val_metrics['macro_f1']:.3f}")

   LR (C=0.01): acc=0.320  F1=0.296


    LR (C=0.1): acc=0.412  F1=0.364


    LR (C=1.0): acc=0.440  F1=0.379


   LR (C=10.0): acc=0.448  F1=0.377


In [10]:
# Multinomial NB sweep
for alpha in [0.01, 0.1, 1.0]:
    name = f"NB (α={alpha})"
    m = train_model(Xtr, y_train, MultinomialNB, alpha=alpha)
    val_metrics = evaluate_model(m, Xva, y_val)
    results.append({
        "model": "MultinomialNB",
        "params": f"alpha={alpha}",
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
    })
    fitted[name] = m
    print(f"{name:>14}: acc={val_metrics['accuracy']:.3f}  F1={val_metrics['macro_f1']:.3f}")

   NB (α=0.01): acc=0.527  F1=0.363
    NB (α=0.1): acc=0.533  F1=0.363


    NB (α=1.0): acc=0.571  F1=0.343


### All 7 configs, side by side

Sorted by val macro F1. The top row is the model evaluated on test next.

In [11]:
summary = pd.DataFrame(results).sort_values("val_macro_f1", ascending=False).reset_index(drop=True)
summary.round(3)

,model,params,val_accuracy,val_macro_f1
0,LogisticRegression,"C=1.0, penalty=l2, class_weight=balanced",0.440,0.379
1,LogisticRegression,"C=10.0, penalty=l2, class_weight=balanced",0.448,0.377
2,LogisticRegression,"C=0.1, penalty=l2, class_weight=balanced",0.412,0.364
3,MultinomialNB,alpha=0.01,0.527,0.363
4,MultinomialNB,alpha=0.1,0.533,0.363
5,MultinomialNB,alpha=1.0,0.571,0.343
6,LogisticRegression,"C=0.01, penalty=l2, class_weight=balanced",0.320,0.296


## Best Model on Test

Pick the winner by val macro F1, then score it on the held-out test set (first time that data gets touched). Reporting 4 metrics: overall accuracy, macro F1, and per-class precision + recall — the per-class numbers are useful because the class imbalance means overall accuracy can hide a model that's failing on Calm or Angry.

In [12]:
# reconstruct the name of the winning config and look it up in `fitted`
top = summary.iloc[0]
if top["model"] == "LogisticRegression":
    C = float(top["params"].split(",")[0].split("=")[1])
    best_name = f"LR (C={C})"
else:
    alpha = float(top["params"].split("=")[1])
    best_name = f"NB (α={alpha})"

print(f"best on val: {best_name}  (val macro F1 = {top['val_macro_f1']:.3f})")
best_model = fitted[best_name]

best on val: LR (C=1.0)  (val macro F1 = 0.379)


In [13]:
test_metrics = evaluate_model(best_model, Xte, y_test)

print(f"test accuracy : {test_metrics['accuracy']:.3f}")
print(f"test macro F1 : {test_metrics['macro_f1']:.3f}")
print()
print("per-class precision:")
for cls, p in test_metrics["per_class_precision"].items():
    print(f"  {cls:<9} {p:.3f}")
print()
print("per-class recall:")
for cls, r in test_metrics["per_class_recall"].items():
    print(f"  {cls:<9} {r:.3f}")

test accuracy : 0.432
test macro F1 : 0.371

per-class precision:
  Angry     0.228
  Calm      0.274
  Hype      0.731
  Romantic  0.260
  Sad       0.331

per-class recall:
  Angry     0.448
  Calm      0.391
  Hype      0.462
  Romantic  0.404
  Sad       0.362


## LR vs NB — head to head

Picking the best of each family (by val macro F1) and scoring both on the test set. Same splits, same metrics, so this is a fair architecture comparison.

In [14]:
lr_best_row = summary[summary["model"] == "LogisticRegression"].iloc[0]
nb_best_row = summary[summary["model"] == "MultinomialNB"].iloc[0]

C_lr = float(lr_best_row["params"].split(",")[0].split("=")[1])
alpha_nb = float(nb_best_row["params"].split("=")[1])

lr_model = fitted[f"LR (C={C_lr})"]
nb_model = fitted[f"NB (α={alpha_nb})"]

lr_test = evaluate_model(lr_model, Xte, y_test)
nb_test = evaluate_model(nb_model, Xte, y_test)

compare = pd.DataFrame([
    {
        "model": f"LR (C={C_lr})",
        "val_macro_f1": lr_best_row["val_macro_f1"],
        "test_accuracy": lr_test["accuracy"],
        "test_macro_f1": lr_test["macro_f1"],
    },
    {
        "model": f"NB (α={alpha_nb})",
        "val_macro_f1": nb_best_row["val_macro_f1"],
        "test_accuracy": nb_test["accuracy"],
        "test_macro_f1": nb_test["macro_f1"],
    },
])
compare.round(3)

,model,val_macro_f1,test_accuracy,test_macro_f1
0,LR (C=1.0),0.379,0.432,0.371
1,NB (α=0.01),0.363,0.525,0.356


## Save best model + Vectorizer

Later notebooks (03_evaluation) and the Streamlit app both need to load these — saving to `models/`. Both get pickled so SHAP can re-open the same sklearn objects.

In [15]:
joblib.dump(best_model, "../models/best_classifier.pkl")
joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")
print("saved best_classifier.pkl and tfidf_vectorizer.pkl")

saved best_classifier.pkl and tfidf_vectorizer.pkl
